# Machine Markets — advanced research run

**Your task:** run every cell, download the last ZIP and upload it in our chat.
Deep Research is selected. The notebook carries its own checked source code.

**Research question:** can information about a market's aggregate response improve
a financial exposure decision after information and trading costs, even when the
underlying trading cohorts are difficult to identify?

We model a desk paid to hold inventory for a fixed period. This is a controlled
research experiment, not a strategy promising investment profits. The original
papers remain the papers; this notebook does not generate an empty replacement.

## What will run

1. Mathematical/accounting tests and an exact positive-value numerical witness.
2. Lean verification of **seven concrete scalar inequalities**. The market
   dynamics, probability arguments and full paper are not Lean-formalized.
3. 288 cohort economies, 24 seed clusters and 4,096 measurement repetitions for
   each economy and each N=8,32,128. Horizons are 10,60,240 model minutes.
4. Seven measurement rules, an oracle and a matched-error no-data benchmark,
   with execution, carry and information charges. Negative results are retained.
5. Slow memory, correlated noise, heavy tails, biased observations and future
   regime changes. The last two deliberately challenge the assumptions.
6. Optional event-level quote-data forecasts and an isolated exposure replay.
   Without suitable data that stage is explicitly **NOT_RUN**.

Confidence intervals describe this simulator design. The run cannot establish
first-in-literature novelty, independent replication, or a 9+ paper rating.

In [ ]:
#@title 1. Research settings (defaults are ready)
import os,sys,json,hashlib,subprocess,shutil,zipfile,io,base64,platform,time
from pathlib import Path

PROFILE = "deep" #@param ["deep", "research", "quick"]
BASE_SEED = 20261101 #@param {type:"integer"}
WORKERS = 0 #@param {type:"integer"}
RUN_LEAN = True #@param {type:"boolean"}
USE_DRIVE = False #@param {type:"boolean"}
RUN_MARKET = False #@param {type:"boolean"}
MARKET_CSV = "" #@param {type:"string"}
MARKET_METADATA = "" #@param {type:"string"}

# Keep these unchanged to resume. Change RUN_LABEL after any source, environment,
# profile or seed change. Never run two copies against the same output folder.
RUN_LABEL = "MM_advanced_deep_seed20261101_v1" #@param {type:"string"}
if not RUN_LABEL or Path(RUN_LABEL).name != RUN_LABEL or "." in RUN_LABEL:
    raise ValueError("Use a simple run label without slashes or dots")
if sys.version_info < (3,11):
    raise RuntimeError("Python 3.11 or newer is required")
BASE = Path(os.environ.get("MM_WORK_ROOT", "/content/machine_markets"))
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/MachineMarketsResearch")
else:
    RESULTS_ROOT = BASE / "results"
RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
RUN_DIR = RESULTS_ROOT / RUN_LABEL
SETUP_LOG = RESULTS_ROOT / (RUN_LABEL+"_setup.log")
DRIVER_LOG = RESULTS_ROOT / (RUN_LABEL+"_driver.log")
# Large array operations run in worker processes. Avoid nested BLAS thread pools.
RUN_ENV = dict(os.environ,OMP_NUM_THREADS="1",OPENBLAS_NUM_THREADS="1",MKL_NUM_THREADS="1",NUMEXPR_NUM_THREADS="1")
print("Profile:",PROFILE,"| Python:",platform.python_version(),"| Results:",RUN_DIR)
print("CPU vector workloads; a GPU is not required. Worker count 0 uses available CPU quota.")
print("Enable USE_DRIVE before running for checkpoints that survive a runtime reset.")

In [ ]:
#@title 2. Restore and verify the embedded source
SOURCE_SHA256 = "01e0ec0efc1a0c11b9e031be7f81fd8f688eb62d0f36b4b64bdf8c4a36fa604e"
SOURCE_B64 = "..."
raw = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256, "Embedded source checksum failed"
SRC = BASE / ("source_"+SOURCE_SHA256[:16])
SRC.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        p=Path(name)
        if p.is_absolute() or ".." in p.parts:
            raise ValueError("Invalid embedded source path")
        (SRC/p).write_bytes(z.read(name))
(SRC/"EMBEDDED_SOURCE.json").write_text(json.dumps({"archive_sha256":SOURCE_SHA256},indent=2))
print("Verified source:",SOURCE_SHA256,"|",len(list(SRC.iterdir())),"files")

In [ ]:
#@title 3. Install the tested numerical environment
# Installation happens in a subprocess. The research runner starts a NEW Python
# process, so it uses the installed versions even if Colab had imported old ones.
with SETUP_LOG.open("w") as log:
    if os.environ.get("MM_SKIP_INSTALL") != "1":
        p=subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check","-r",str(SRC/"requirements.txt")],stdout=log,stderr=subprocess.STDOUT,text=True)
        if p.returncode:
            print(SETUP_LOG.read_text()[-8000:])
            raise RuntimeError("Dependency installation failed; setup log retained")
    if RUN_LEAN and shutil.which("zstd") is None:
        for command in [["apt-get","-qq","update"],["apt-get","-qq","install","-y","zstd"]]:
            p=subprocess.run(command,stdout=log,stderr=subprocess.STDOUT,text=True)
            if p.returncode:
                print(SETUP_LOG.read_text()[-8000:])
                raise RuntimeError("Lean archive support installation failed; log retained")
probe=subprocess.run([sys.executable,"-c","import numpy,scipy,pandas,matplotlib;print('numpy',numpy.__version__,'scipy',scipy.__version__,'pandas',pandas.__version__,'matplotlib',matplotlib.__version__)"],capture_output=True,text=True,env=RUN_ENV)
print(probe.stdout or probe.stderr)
if probe.returncode:
    raise RuntimeError("Numerical environment probe failed")
print("Setup complete. Lean's pinned compiler downloads once (approximately 344 MB).")

## Optional market data

Leave RUN_MARKET=False for the complete synthetic/proof audit. To add your own
market data, put the CSV and metadata JSON in Drive, set their absolute paths
above and enable RUN_MARKET **before running**. No account keys are needed.

Required CSV: timestamp (ISO-8601 with timezone), market, bid, ask, bid_size,
ask_size. Rows must contain complete best-quote states after events, in time
order. Declare source, venue, instrument type, units, prior data use, taker fees
and carry in metadata; inspect `market_metadata.example.json` in the code ZIP.
Do not submit candles or sparse quote snapshots as complete order-book events.
The original data are not copied into the final ZIP; keep them for replication.

This stage uses a past-only response **score**, purged train/validation/test
folds and EWMA/HAR/OFI benchmarks. It does not turn public quotes into the
theorem's independent reset experiments or prove market safety.

In [ ]:
#@title 4. Run all checks and experiments
cmd=[sys.executable,str(SRC/"runner.py"),"--out",str(RUN_DIR),"--profile",PROFILE,"--base-seed",str(BASE_SEED),"--workers",str(WORKERS),"--lean-cache",str(BASE/"lean_cache")]
if not RUN_LEAN:
    cmd.append("--skip-lean")
if RUN_MARKET:
    if not Path(MARKET_CSV).is_file() or not Path(MARKET_METADATA).is_file():
        raise ValueError("Supply both existing MARKET_CSV and MARKET_METADATA files, or disable RUN_MARKET")
    cmd += ["--market-csv",MARKET_CSV,"--market-metadata",MARKET_METADATA]
print("Starting:",PROFILE,"| checkpoints:",RUN_DIR)
with DRIVER_LOG.open("w") as log:
    p=subprocess.Popen(cmd,cwd=SRC,env=RUN_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        print(line,end="",flush=True);log.write(line);log.flush()
    RUN_RETURN_CODE=p.wait()
print("Runner exit code:",RUN_RETURN_CODE)
if RUN_RETURN_CODE:
    print("A stage failed. Keep running the remaining cells and return the ZIP with the failure logs.")

In [ ]:
#@title 5. Review the status and main tables
from IPython.display import display,HTML,Image
import csv,html
status_path=RUN_DIR/"run_status.json"
if status_path.exists():
    STATUS=json.loads(status_path.read_text());print(json.dumps(STATUS,indent=2))
else:
    STATUS={"status":"INCOMPLETE","reason":"No runner status file; inspect driver/setup logs"};print(STATUS)
receipt=RUN_DIR/"exact_witness/lean_receipt.json"
if receipt.exists():
    print("Lean receipt:");print(receipt.read_text())
print("Lean checks the concrete inequalities only. It does not certify the whole paper.")
for name in ["scenario_policy_summary.csv","seed_cluster_intervals.csv"]:
    p=RUN_DIR/name
    if p.exists():
        rows=list(csv.reader(p.open()))
        table="<table style='border-collapse:collapse;font-size:12px'>"+"".join("<tr>"+"".join("<td style='border:1px solid #ccc;padding:4px'>"+html.escape(v)+"</td>" for v in row)+"</tr>" for row in rows)+"</table>"
        display(HTML("<h3>"+name+"</h3>"+table))
for name in ["net_value_by_scenario.png","coverage_failures.png","identification_scaling.png","information_cost_frontier.png"]:
    if (RUN_DIR/name).exists():display(Image(filename=str(RUN_DIR/name)))

## How to read the output

- A positive net advantage means that a *specified* information purchase improved
  expected service profit relative to the matched-error no-data benchmark in
  this model. A negative value is a real result, not a row to discard.
- A valid contract needs its stated noise, reset and stationarity assumptions.
  Apparent extra profit from an invalid method is not a safe improvement.
- A non-rejection in the binomial audit is not proof of a safety theorem.
- Strong performance on generated markets is not proof that the source papers
  are novel, correctly proved in full, or ready for publication.
- The market stage and specialist novelty/proof review are separate obligations.

The returned ZIP lets us identify which claims survive, which fail, and what
can responsibly be integrated into the existing manuscripts.

In [ ]:
#@title 6. Package every result and download the ZIP
# Run this cell even after an experiment error. It also packages partial runs.
RUN_DIR.mkdir(parents=True,exist_ok=True)
for p,name in [(SETUP_LOG,"notebook_setup.log"),(DRIVER_LOG,"notebook_driver.log")]:
    if p.exists():shutil.copy2(p,RUN_DIR/name)
(RUN_DIR/"notebook_receipt.json").write_text(json.dumps({
    "source_archive_sha256":SOURCE_SHA256,"profile":PROFILE,"base_seed":BASE_SEED,
    "runner_returncode":globals().get("RUN_RETURN_CODE"),
    "completed_by_notebook":globals().get("RUN_RETURN_CODE")==0,
    "market_requested":RUN_MARKET,"full_paper_proved":False,"novelty_confirmed":False
},indent=2))
manifest={str(p.relative_to(RUN_DIR)):hashlib.sha256(p.read_bytes()).hexdigest()
          for p in sorted(RUN_DIR.rglob("*")) if p.is_file() and p.name!="manifest.json" and not p.name.endswith(".tmp")}
(RUN_DIR/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
ZIP_PATH=Path(str(RUN_DIR)+".zip")
with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for name in manifest:z.write(RUN_DIR/name,name)
    z.write(RUN_DIR/"manifest.json","manifest.json")
with zipfile.ZipFile(ZIP_PATH) as z:
    for name,h in manifest.items():
        assert hashlib.sha256(z.read(name)).hexdigest()==h,name
print("Verified",len(manifest),"files | ZIP MB:",round(ZIP_PATH.stat().st_size/1e6,2))
print("ZIP SHA256:",hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print("Upload this ZIP in our chat:",ZIP_PATH.name)
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print("Outside Colab. Download:",ZIP_PATH)